In [1]:
pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


# Producer 

In [1]:
import random
import json
import time
import pandas as pd
from kafka import KafkaProducer

# ==========================================
# 1. CONFIGURACIÓN DEL SIMULADOR
# ==========================================
KAFKA_BROKER = 'kafka:9093'
KAFKA_TOPIC = 'transactions'
CSV_PATH = '/home/jovyan/work/data/creditcard.csv'

# ==========================================
# 2. INICIALIZAR EL PRODUCTOR KAFKA
# ==========================================
print(f"Conectando al broker Kafka en {KAFKA_BROKER}...")
producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    # Optimizaciones extra para enviar datos en ráfagas eficientes
    linger_ms=5,        # Kafka espera 5ms para agrupar varios mensajes antes de enviar
    batch_size=32768    # Tamaño máximo del paquete (en bytes)
)
print("¡Conexión establecida!")

# ==========================================
# 3. PREPARACIÓN DE DATOS (EL SECRETO DEL RENDIMIENTO)
# ==========================================
print(f"Cargando dataset desde {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

# Quitamos la columna 'Class' y separamos
df_fraud = df[df['Class'] == 1].drop(columns=['Class'])
df_normal = df[df['Class'] == 0].drop(columns=['Class'])

print("Convirtiendo datos a listas de diccionarios en memoria...")
# Esto elimina el cuello de botella de Pandas en el bucle
fraud_records = df_fraud.to_dict(orient='records')
normal_records = df_normal.to_dict(orient='records')

print(f"Dataset cargado. Fraudes: {len(fraud_records)} | Normales: {len(normal_records)}")
print("Iniciando inyección de datos de ALTO RENDIMIENTO (Presiona 'Stop' o Ctrl+C para detener)...")
print("-" * 50)

# ==========================================
# 4. BUCLE DE SIMULACIÓN (STRESS TEST)
# ==========================================
count = 0
start_time = time.time()

try:
    while True:
        # 1. Selección ultrarrápida usando librerías nativas de Python
        if random.random() < 0.2 and fraud_records:
            payload = random.choice(fraud_records)
        else:
            payload = random.choice(normal_records)

        # 2. Envío a Kafka (completamente asíncrono, sin flush)
        producer.send(KAFKA_TOPIC, value=payload)
        count += 1
        
        # 3. Informar por pantalla solo cada 5000 registros para no bloquear el bucle
        if count % 5000 == 0:
            elapsed = time.time() - start_time
            print(f"Enviados {count} registros... Tasa actual: {5000/elapsed:.0f} transacciones/seg")
            start_time = time.time() # Reseteamos el reloj

        # El sleep está comentado. Si tu PC se satura mucho, descomenta la línea de abajo.
        # time.sleep(0.0001)

except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
except Exception as e:
    print(f"\nError en el simulador: {e}")
finally:
    print("Vaciando buffers y enviando registros pendientes a Kafka...")
    producer.flush()
    producer.close()
    print(f"Total de registros enviados al clúster: {count}")

Conectando al broker Kafka en kafka:9093...
¡Conexión establecida!
Cargando dataset desde /home/jovyan/work/data/creditcard.csv...
Convirtiendo datos a listas de diccionarios en memoria...
Dataset cargado. Fraudes: 492 | Normales: 284315
Iniciando inyección de datos de ALTO RENDIMIENTO (Presiona 'Stop' o Ctrl+C para detener)...
--------------------------------------------------
Enviados 5000 registros... Tasa actual: 6503 transacciones/seg
Enviados 10000 registros... Tasa actual: 6840 transacciones/seg
Enviados 15000 registros... Tasa actual: 3983 transacciones/seg
Enviados 20000 registros... Tasa actual: 5065 transacciones/seg
Enviados 25000 registros... Tasa actual: 5813 transacciones/seg
Enviados 30000 registros... Tasa actual: 4250 transacciones/seg
Enviados 35000 registros... Tasa actual: 4484 transacciones/seg
Enviados 40000 registros... Tasa actual: 5273 transacciones/seg
Enviados 45000 registros... Tasa actual: 6255 transacciones/seg
Enviados 50000 registros... Tasa actual: 666

## 1. Instalar el cliente de Kafka dentro de Jupyter
!pip install kafka-python-ng pandas

import time
import json
import pandas as pd
from kafka import KafkaProducer

